In [15]:
from tensorflow.keras.datasets import imdb
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Embedding, Dense

# Load data
(X_train, y_train), (X_test, y_test) = imdb.load_data(num_words=5000)

# Pad sequences
X_train = pad_sequences(X_train, maxlen=200)
X_test = pad_sequences(X_test, maxlen=200)

# Model
model = Sequential()
model.add(Embedding(5000, 64))
model.add(LSTM(64))
model.add(Dense(1, activation='sigmoid'))

model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])

# Train
model.fit(X_train, y_train, epochs=3, batch_size=64)

# Evaluate
loss, acc = model.evaluate(X_test, y_test)
print("Accuracy:", acc)

Epoch 1/3
391/391 ━━━━━━━━━━━━━━━━━━━━ 58s 142ms/step - accuracy: 0.7991 - loss: 0.4335
Epoch 2/3
391/391 ━━━━━━━━━━━━━━━━━━━━ 55s 140ms/step - accuracy: 0.8890 - loss: 0.2767
Epoch 3/3
391/391 ━━━━━━━━━━━━━━━━━━━━ 81s 136ms/step - accuracy: 0.9079 - loss: 0.2359
782/782 ━━━━━━━━━━━━━━━━━━━━ 23s 29ms/step - accuracy: 0.8628 - loss: 0.3213
Accuracy: 0.8628000020980835


In [14]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Input

# Load dataset
df = pd.read_excel('airline-passengers.csv')

# The Excel file seems to contain comma-separated values within a single column.
# We need to split the single column into 'Month' and 'Passengers'.
# First, let's ensure the column name is treated as a string and not part of the data.
# The current column name is something like '"Month" ,"Passengers"'.
# Let's assume the first row of data contains the actual header if it's not the case already.

# Strip quotes and spaces from the column name and first data entry
df.columns = [col.strip('" ').replace('"', '') for col in df.columns]

# If there's only one column and its name is composite, we need to split it.
if len(df.columns) == 1 and ',' in df.columns[0]:
    # The column name itself might be the combined string. Let's get the original raw header.
    # For this specific file, it's safer to treat the first row as data and parse headers.
    # Reloading with no header to correctly parse column names from the first row.
    df = pd.read_excel('airline-passengers.csv', header=None)

    # The first row contains the actual headers, which are comma-separated within the first cell.
    # Split the first cell to get proper column names.
    header_str = df.iloc[0, 0].strip('" ').replace('"', '')
    column_names = [name.strip() for name in header_str.split(',')]

    # Assign the new column names
    df.columns = [column_names[0] + ', ' + column_names[1]] # Temporarily assign a unique name for the single column

    # Split the single column into two and reassign proper names
    df_split = df[df.columns[0]].str.split(',', expand=True)
    df_split.columns = column_names

    # Drop the first row as it contained the header information
    df = df_split.iloc[1:].copy()

    # Convert 'Passengers' column to numeric type
    df['Passengers'] = pd.to_numeric(df['Passengers'])

data = df['Passengers'].values.reshape(-1,1)

# Normalize
scaler = MinMaxScaler()
data = scaler.fit_transform(data)

# Create sequences
X, y = [], []
for i in range(10, len(data)):
    X.append(data[i-10:i])
    y.append(data[i])

X, y = np.array(X), np.array(y)

# Split data
split = int(0.8 * len(X))
X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]

# Build LSTM model
model = Sequential()
model.add(Input(shape=(X.shape[1], 1)))
model.add(LSTM(50, activation='relu'))
model.add(Dense(1))

model.compile(optimizer='adam', loss='mse')

# Train
model.fit(X_train, y_train, epochs=20, batch_size=16)

# Predict
predictions = model.predict(X_test)
predictions = scaler.inverse_transform(predictions)

print("Predictions:", predictions[:5])

Epoch 1/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 4s 26ms/step - loss: 0.0980
Epoch 2/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step - loss: 0.0604
Epoch 3/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step - loss: 0.0312
Epoch 4/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step - loss: 0.0156
Epoch 5/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - loss: 0.0133
Epoch 6/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step - loss: 0.0102
Epoch 7/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step - loss: 0.0096
Epoch 8/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - loss: 0.0079
Epoch 9/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - loss: 0.0074
Epoch 10/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step - loss: 0.0071
Epoch 11/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step - loss: 0.0067
Epoch 12/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step - loss: 0.0068
Epoch 13/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 1s 61ms/step - loss: 0.0067
Epoch 14/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step - loss: 0.0066
Epoch 15/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step - loss: 0.0071
Epoch 16/20
7/7 ━━━━━━━━━━━━━━━━━━